In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "global_settings.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
from src.global_settings import settings


In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.lines import Line2D
from sklearn.preprocessing import MultiLabelBinarizer

import seaborn as sns

from src.global_settings import settings
from src.features import load_score_data, load_sample_records

from src.sae import SparseAutoencoder
from src.activations import ActivationDataset
from src.utils.model_loader import load_sae

### (1) 특징 발현

In [ ]:
META_DIR = settings.feature_meta_dir
RECORD_DIR = settings.feature_record_dir

In [ ]:
checkpoint = "f3072_k8_alpha"
layer = 21

scores_total = load_score_data(checkpoint=checkpoint, layer=layer, metric_name="total_post")
score_avg = load_score_data(checkpoint=checkpoint, layer=layer, metric_name="avg_pre")
df_scores_total = pd.DataFrame(scores_total)
df_scores_total.columns = [f"feat_{i}" for i in df_scores_total.columns]

df_scores_total

In [ ]:
df_scores_avg = pd.DataFrame(score_avg)
df_scores_avg.columns = [f"feat_{i}" for i in df_scores_avg.columns]

df_scores_avg

In [ ]:
def min_max_norm(df: pd.DataFrame) -> pd.DataFrame:
    df_scaled = (df - df.min()) / (df.max() - df.min())
    df_scaled = df_scaled.fillna(0.0)
    return df_scaled

df_scores_avg = min_max_norm(df_scores_avg)

df_scores_avg

In [ ]:
sample_indices = np.random.choice(df_scores_avg.index, size=100, replace=False)
feature_columns = np.random.choice(df_scores_avg.columns, size=100, replace=False)
df_sample = df_scores_avg.loc[sample_indices, feature_columns]
df_sample = df_sample.sort_index()

plt.figure(figsize=(20, 8))
sns.heatmap(df_sample, cmap="viridis")
plt.title(f"Feature Scores (100 random features)")
plt.xlabel("Feature Index")
plt.ylabel("Sample Index")
plt.xticks(rotation=45)
plt.show()

### (2) Major Features

In [ ]:
df_scores_total.columns = [i for i in range(df_scores_total.shape[1])]

feature_active_counts = (df_scores_total > 0).sum(axis=0)
feature_active_counts = feature_active_counts[feature_active_counts > 0]

In [ ]:
colors = ["red" if v <= 200 else "steelblue" for v in feature_active_counts.sort_values(ascending=False)]

plt.figure(figsize=(20, 6))
ax = feature_active_counts.sort_values(ascending=False).plot(kind="bar", color=colors, alpha=0.7, width=1.0)
plt.title("Feature Activation Frequency")
plt.xlabel("Feature Index")
plt.ylabel("Active Samples")
ax.xaxis.set_major_locator(MaxNLocator(nbins=50))
plt.xticks(rotation=45)
plt.grid(True, axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
major_features_indices = feature_active_counts[feature_active_counts > 200].index.tolist()

len(major_features_indices)

In [ ]:
df_major_scores = df_scores_total[major_features_indices]

df_major_scores

### (3) 특징의 방향성

In [ ]:
df_active_major = df_major_scores > 0

active_features_series = df_active_major.apply(lambda row: row[row].index.tolist(), axis=1)

df_active_feature_list_per_sample = active_features_series.to_frame(name='active_features')
df_active_feature_list_per_sample['sample_id'] = df_active_feature_list_per_sample.index
df_active_feature_list_per_sample = df_active_feature_list_per_sample[['sample_id', 'active_features']]

df_active_feature_list_per_sample

In [ ]:
text_df = pd.read_csv(settings.text_dataset_path)

pcc_score_label = df_major_scores.corrwith(text_df["label"], method="pearson")
pcc_score_pred = df_major_scores.corrwith(text_df["pred"], method="pearson")

df_corr = pd.DataFrame({
    "pcc_label": pcc_score_label,
    "pcc_pred": pcc_score_pred
})

In [ ]:
df_corr

In [ ]:
df_corr = df_corr.sort_values(by="pcc_label")
df_corr['color'] = np.where(df_corr['pcc_label'] < 0, '#1f77b4', '#d62728') # (human: blue, ai: red)

df_corr

In [ ]:
df_sorted = df_corr.sort_values(by="pcc_label")

N = 100  # Human-side 20개, AI-side 20개 (총 40개)
df_plot = pd.concat([
    df_sorted.head(N),  # Human-side Top N
    df_sorted.tail(N)   # AI-side Top N
])
df_plot['color'] = np.where(df_plot['pcc_label'] < 0, '#1f77b4', '#d62728')

plt.figure(figsize=(10, 8))

plt.scatter(
    df_plot['pcc_label'],
    df_plot.index,
    color=df_plot['color'],
    edgecolor='black',
    alpha=0.7,
    linewidth=1.5,
    s=100,
    zorder=5,
    label='pcc_label'
)

plt.scatter(
    df_plot['pcc_pred'],
    df_plot.index,
    color=df_plot['color'],
    edgecolor='gray',
    linewidth=1.5,
    s=100,
    zorder=5,
    alpha=0.5,
    label='pcc_pred'
)

plt.title('Feature Score - Pearson Correlation with $label$ and $pred$', fontsize=14)
plt.xlabel('Pearson Correlation ($r$) (0=Human, 1=AI)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.axvline(x=0, color='grey', linestyle='--', linewidth=0.5)

plt.text(0.02, 0.98, '← Human-side (label=0)', 
         transform=plt.gca().transAxes, ha='left', va='top', fontsize=12, 
         color='#1f77b4', weight='bold')
plt.text(0.9, 0.95, 'AI-side (label=1) →', 
         transform=plt.gca().transAxes, ha='right', va='bottom', fontsize=12, 
         color='#d62728', weight='bold')

legend = [
    Line2D([0], [0],
           marker='o',
           color='gray',
           markerfacecolor='gray',
           markeredgecolor='black',
           markersize=10,
           linestyle='None',
           alpha=0.7,
           label='pcc_label'),
    Line2D([0], [0],
           marker='o',
           color='lightgray',
           markerfacecolor='lightgray',
           markeredgecolor='gray',
           markersize=10,
           linestyle='None',
           alpha=0.5,
           label='pcc_pred')
]
plt.legend(handles=legend, loc="upper center")

plt.tight_layout()
plt.show()

In [ ]:
mlb = MultiLabelBinarizer()
active_matrix_np = mlb.fit_transform(df_active_feature_list_per_sample['active_features'])
df_active_matrix = pd.DataFrame(
    active_matrix_np,
    columns=mlb.classes_,
    index=df_active_feature_list_per_sample.index
)
pcc_label_freq = df_active_matrix.corrwith(text_df["label"], method="pearson")
pcc_pred_freq = df_active_matrix.corrwith(text_df["pred"], method="pearson")

df_corr_freq = pd.DataFrame({
    "pcc_label": pcc_label_freq,
    "pcc_pred": pcc_pred_freq
})

plt.figure(figsize=(10, 8))

sns.scatterplot(
    data=df_corr_freq,
    x='pcc_label', 
    y='pcc_pred', 
    alpha=0.5,
    s=100,
)

plt.title('Feature Frequency - Pearson Correlation with $label$ and $pred$', fontsize=16)
plt.xlabel('Correlation with True Label ($pcc_{{label}}$)', fontsize=12)
plt.ylabel('Correlation with Model Prediction ($pcc_{{pred}}$)', fontsize=12)

plt.axvline(x=0, color='grey', linestyle='--', linewidth=0.5)
plt.axhline(y=0, color='grey', linestyle='--', linewidth=0.5)

plt.text(0.95, 0.95, 'Proper AI-Feature\n(label ↑, pred ↑)', 
         transform=plt.gca().transAxes, ha='right', va='top', fontsize=10, color='red')
plt.text(0.05, 0.05, 'Proper Human-Feature\n(label ↓, pred ↓)', 
         transform=plt.gca().transAxes, ha='left', va='bottom', fontsize=10, color='blue')
plt.text(0.05, 0.95, 'AI-Confusing Feature\n(label ↓, pred ↑)', 
         transform=plt.gca().transAxes, ha='left', va='top', fontsize=10, color='orange')
plt.text(0.95, 0.05, 'Human-Confusing Feature\n(label ↑, pred ↓)', 
         transform=plt.gca().transAxes, ha='right', va='bottom', fontsize=10, color='orange')

plt.tight_layout()
plt.show()

In [ ]:
BS = 128
ACTIVATION_DATASET_DIR_TEMPLATE = (
    "./data/post_mlp_residual_stream/layer_{layer_idx}"
)
DTYPE = torch.float32

META_DIR = settings.feature_meta_dir
RECORD_DIR = settings.feature_record_dir

In [ ]:
def init_dataloader(layer_index) -> DataLoader:
    dataset = ActivationDataset(data_dir=ACTIVATION_DATASET_DIR_TEMPLATE.format(layer_idx=layer_index), dtype=DTYPE)
    dataloader = DataLoader(dataset, batch_size=BS, shuffle=False, num_workers=4, pin_memory=True)
    return dataloader

text_df = pd.read_csv(settings.text_dataset_path)

# 1. 특징 점수 계산

In [ ]:
model_tag = "f3072_k256"
hparamset_tag = "alpha"
layer_index = 1
checkpoint = f"{model_tag}_{hparamset_tag}"
dataloader = init_dataloader(layer_index)

model: SparseAutoencoder = load_sae(
    trained_model_path=None,
    model_tag=model_tag,
    hparamset_tag=hparamset_tag,
    layer_idx=layer_index,
    return_ckpt=False
)

In [ ]:
record_dir = Path(RECORD_DIR) / checkpoint / str(21)

pos_samples = load_sample_records(record_dir, feature_index=4, sample_type="positive")
neg_samples = load_sample_records(record_dir, feature_index=4, sample_type="negative")

In [ ]:
neg_samples.iloc[0]["active_tokens"]

In [ ]:
pos_samples.iloc[0]["active_tokens"]

In [ ]:
score_dir = Path(META_DIR) / checkpoint / str(layer_index)

# scores_total_pre = load_score_data(score_dir, "total_pre")
scores_avg_pre = load_score_data(score_dir, "avg_pre")
scores_total_post = load_score_data(score_dir, "total_post")
# scores_avg_post = load_score_data(score_dir, "avg_post")

In [ ]:
scores_avg_pre_feat_0 = scores_avg_pre[:, 0]
scores_total_post_feat_0 = scores_total_post[:, 0]

random_sample_indices = np.random.choice(len(scores_avg_pre_feat_0), size=100, replace=False)
scores_avg_pre_feat_0_sampled = scores_avg_pre_feat_0[random_sample_indices]
scores_total_post_feat_0_sampled = scores_total_post_feat_0[random_sample_indices]

plt.figure(figsize=(10, 6))
plt.scatter(random_sample_indices, scores_avg_pre_feat_0_sampled, alpha=0.5)
plt.scatter(random_sample_indices, scores_total_post_feat_0_sampled, alpha=0.5)
plt.grid(True)
plt.show()

## 1-1. Post-activation 버전(top-k)

In [ ]:
feature_indices = range(3072)
device = torch.device("cuda")
use_normalization = False
activation_fn = "topk"
activation_k = 256

model.to(device)
model.eval()

sample_scores, sample_labels, sample_preds = compute_sample_feature_scores(
    model=model,
    dataloader=dataloader,
    text_df=text_df,
    checkpoint=checkpoint,
    layer=layer_index,
    feature_indices=feature_indices,
    device=device,
    use_normalization=use_normalization,
    activation_fn=activation_fn,
    activation_k=activation_k,
)

In [ ]:
scores_df = pd.DataFrame(sample_scores, columns=[f"feature_{idx}" for idx in feature_indices])

In [ ]:
score_map_path = os.path.join(META_DIR, checkpoint, str(layer_index), "score_map.npy")

score_map_np = scores_df.to_numpy()
np.save(score_map_path, score_map_np)

In [ ]:
score_df_loaded_from_npy = pd.DataFrame(
    np.load(score_map_path),
    columns=[f"feature_{idx}" for idx in feature_indices]
)

score_df_loaded_from_npy

In [ ]:
activate_counts = {
    idx: np.sum(scores_df[f"feature_{idx}"] > 0) for idx in feature_indices
}

### (1) 특징별 활성화하는 샘플의 비율 확인해보기

In [ ]:
plt.figure(figsize=(20, 10))
plt.scatter(feature_indices[:1000], list(activate_counts.values())[:1000])
plt.title(f"Activated Sample Counts per Feature (ckpt: {checkpoint}, layer: {layer_index})")
plt.xlabel("Feature Index")
plt.ylabel("Activation Count")
plt.show()

In [ ]:
non_active_features = [idx for idx, count in activate_counts.items() if count < 200]

len(non_active_features)

### (2) 모델 예측과의 상관성 계산해보기

In [ ]:
preds = text_df["pred"].tolist()

corrs = []
# pred-score correlation analysis
for feat_idx in scores_df.columns:
    scores = scores_df[feat_idx]
    corr = np.corrcoef(scores, preds)[0, 1]
    corrs.append(corr)

In [ ]:
plt.figure(figsize=(20, 6))
plt.bar(feature_indices, corrs)
plt.title(f"Prediction-Score Correlation per Feature (ckpt: {checkpoint}, layer: {layer_index})")
plt.xlabel("Feature Index")
plt.ylabel("Correlation Coefficient")
plt.show()

In [ ]:
# pred-score corr distribution

plt.figure(figsize=(10, 6))
plt.hist(corrs, bins=100)
plt.title(f"Distribution of Prediction-Score Correlation (ckpt: {checkpoint}, layer: {layer_index})")
plt.xlabel("Correlation Coefficient")
plt.ylabel("Frequency")
plt.show()

In [ ]:
positive_corrs = {idx: corr for idx, corr in zip(feature_indices, corrs) if corr > 0.1}
len(positive_corrs)

In [ ]:
negative_corrs = {idx: corr for idx, corr in zip(feature_indices, corrs) if corr < -0.1}
len(negative_corrs)

In [ ]:
# Top 'Significant' features (feature score is bounded to pred)
mask = ~np.isnan(corrs)
valid_indices = np.where(mask)[0]
valid_values = np.array(corrs)[mask]
top_corrs = np.argsort(valid_values)[-15:][::-1]
top_indices = valid_indices[top_corrs]

for feat_idx in top_indices:
    print(f"Feature {feat_idx} (Correlation = {corrs[feat_idx]:.4f})")

In [ ]:
feat_idx = 1444
feat_scores = scores_df[f"feature_{feat_idx}"]

feat_scores_mean = feat_scores.mean()
feat_scores_std = feat_scores.std()
print(f"Min: {feat_scores.min():.4f} \nMax: {feat_scores.max():.4f} \nMean: {feat_scores_mean:.4f} \nStd: {feat_scores_std:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(feat_scores, bins=100)
plt.show()

In [ ]:
BATCH_SIZE = 128
MAX_SEQ_LEN = settings.prev_seq_len  # 1024
tokenizer = AutoTokenizer.from_pretrained(str(settings.lm_weights_path))

text_df = pd.read_csv(str(settings.text_dataset_path))

device = torch.device("cuda")
MODEL_TAG = "f3072_k256"
LAYER_IDX = 21
checkpoint = f"{MODEL_TAG}_alpha_L{LAYER_IDX}"
MODEL_PATH = f"./outputs/weights/{checkpoint}"
sae, sae_ckpt = load_sae(MODEL_PATH, return_ckpt=True)
sae.to(device)

activations_dir = f"./data/post_mlp_residual_stream/layer_{LAYER_IDX}"
dataset = ActivationDataset(data_dir=activations_dir, dtype=torch.float32)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
samples, mask, sample_idxs = next(iter(dataloader))

idx = 4
sample_ts = samples[idx]
sample_mask = mask[idx]
sample_idx = sample_idxs[idx].item()
sample_text = text_df.iloc[sample_idx]["text"]
print("TEXT: ", sample_text)
print("TENSOR: ", sample_ts.shape)

In [ ]:
x = sample_ts.unsqueeze(0).to(device)
preprocessed_input = sae._preprocess_input(x)
encoded = sae.encoder(preprocessed_input)
features = sae._apply_activation(encoded)

encoded_flat = encoded.view(-1).cpu().detach().numpy()
features_flat = features.view(-1).cpu().detach().numpy()

In [ ]:
FEATURE_IDX = 488

In [ ]:
encoded_feature = encoded[0, ...][..., FEATURE_IDX]
encoded_topk_feature = features[0, ...][..., FEATURE_IDX]

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(encoded_feature.cpu().detach().numpy(), label="token acts (before topk)")
plt.plot(encoded_topk_feature.cpu().detach().numpy(), label="token acts (after topk)")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
x = range(len(encoded_feature))
plt.scatter(x, encoded_feature.cpu().detach().numpy(), label="token acts (before topk)")
plt.scatter(x, encoded_topk_feature.cpu().detach().numpy(), label="token acts (after topk)")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(encoded_feature[:200].cpu().detach().numpy(), label="token acts (before topk)")
plt.plot(encoded_topk_feature[:200].cpu().detach().numpy(), label="token acts (after topk)")
plt.legend()
plt.show()

In [ ]:
encoded_flat_norm = (encoded_flat - np.min(encoded_flat)) / (np.max(encoded_flat) - np.min(encoded_flat) + 1e-10)
feature_flat_norm = (features_flat - np.min(features_flat)) / (np.max(features_flat) - np.min(features_flat) + 1e-10)

In [ ]:
plt.figure(figsize=(12, 6))
x = range(len(encoded_flat[:10240]))
plt.scatter(x, encoded_flat[:10240], label="token acts (before topk)")
plt.scatter(x, features_flat[:10240], label="token acts (after topk)")
# plt.plot(encoded_flat_norm[:2048], label="token acts norm (before topk)", linestyle='--')
# plt.plot(feature_flat_norm[:2048], label="token acts norm (after topk)", linestyle='--')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
x = range(len(encoded_flat_norm[:10240]))
plt.scatter(x, encoded_flat_norm[:10240], label="token acts norm (before topk)")
plt.scatter(x, feature_flat_norm[:10240], label="token acts norm (after topk)")
plt.legend()
plt.show()